# Reliability: Strict Builds, Timeouts, and Exceptions

This notebook focuses on operational robustness features from the test suite:
- `strict=True` build validation
- Timeout configuration
- Catching and inspecting Selene exceptions

In [ ]:
import datetime

from guppylang.decorator import guppy
from guppylang.std.quantum import qubit, h, measure
from guppylang.std.builtins import result

from selene_sim.build import build
from selene_sim import Quest
from selene_sim.timeout import Timeout
from selene_sim.exceptions import SeleneTimeoutError, SeleneStartupError, SeleneRuntimeError

## 1) Strict build validation

When `strict=True`, Selene validates intermediate artifacts as well as the user input artifact.

In [ ]:
@guppy
def strict_demo() -> None:
    q0 = qubit()
    h(q0)
    result("c0", measure(q0))

strict_runner = build(strict_demo.compile(), strict=True)
strict_result = list(strict_runner.run(Quest(), n_qubits=1))
print(strict_result)

## 2) Timeout configuration

You can set a single overall timeout or fine-grained timers (`overall`, `per_shot`, `per_result`).

In [ ]:
@guppy
def finite_program() -> None:
    q0 = qubit()
    h(q0)
    result("c0", measure(q0))

timeout_runner = build(finite_program.compile())
shots = list(
    timeout_runner.run_shots(
        Quest(random_seed=88),
        n_qubits=1,
        n_shots=10,
        timeout=Timeout(
            overall=datetime.timedelta(seconds=5),
            per_shot=datetime.timedelta(seconds=1),
            per_result=datetime.timedelta(seconds=1),
        ),
    )
)
print([dict(shot) for shot in shots[:3]])

## 3) Exception handling pattern

In production workflows, catch Selene-specific exceptions and log structured fields (`message`, `stdout`, `stderr`, `code` when available).

In [ ]:
try:
    # Example: force an unrealistically short timeout to demonstrate handling
    _ = list(
        timeout_runner.run_shots(
            Quest(),
            n_qubits=1,
            n_shots=1000,
            timeout=datetime.timedelta(seconds=0),
        )
    )
except SeleneStartupError as e:
    print("Startup error:", e.message)
except SeleneTimeoutError as e:
    print("Timeout error:", e.message)
except SeleneRuntimeError as e:
    print("Runtime error:", e.message)